# E-Commerce Demand Forecasting

## Project Objective
Forecast weekly demand for a selected high-demand e-commerce product and convert the forecast into an inventory replenishment recommendation.

### Business flow
**Historical orders → Completed orders → Demand EDA → Product selection → Weekly forecasting → Inventory planning**


## 1. Business Problem

The business wants to:
- understand historical product demand,
- forecast demand for the next four weeks,
- calculate safety stock and reorder point,
- identify whether current inventory requires replenishment.

The forecasting target is **weekly quantity demanded** for the selected product.


## 2. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Place ecommerce_dataset.csv inside the project's data folder.
DATA_PATH = "ecommerce_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


Rows: 1000123
Columns: 62


## 3. Data Cleaning

The main preparation step is converting `order_date` into a datetime column so that time-based analysis and forecasting can be performed correctly.


In [2]:
df["order_date"] = pd.to_datetime(df["order_date"])

print("Order date type:", df["order_date"].dtype)
print("Duplicate rows:", df.duplicated().sum())

missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]

print("\nColumns with missing values:")
print(missing_values if len(missing_values) else "No missing values")


Order date type: datetime64[us]
Duplicate rows: 0

Columns with missing values:
return_reason        900244
customer_feedback    199617
coupon_code          500083
dtype: int64


## 4. Order Status Analysis

Before forecasting, we examine how orders are distributed across their statuses.

For the forecasting dataset, **Completed** orders are treated as fulfilled sales.


In [20]:
order_status_count = (
    df["order_status"]
    .value_counts()
    .reset_index()
)

order_status_count.columns = [
    "order_status",
    "order_count"
]

print(order_status_count)

fig = px.bar(
    order_status_count,
    x="order_status",
    y="order_count",
    color="order_status",
    text="order_count",
    title="Order Count by Order Status",
    template="plotly_white"
)



fig.update_layout(
    title_x=0.5,
    xaxis_title="Order Status",
    yaxis_title="Number of Orders",
    showlegend=False
)

fig.show()


  order_status  order_count
0    Completed       700232
1     Returned        99879
2      Pending        99748
3    Cancelled        50163
4   Processing        50101


## 5. Create Forecasting Dataset

We keep only **Completed** orders because the project focuses on fulfilled sales demand rather than pending, cancelled, processing, or returned orders.


In [21]:
forecast_df = df[
    df["order_status"] == "Completed"
].copy()

forecast_df["date"] = forecast_df["order_date"].dt.floor("D")

print("Forecasting rows:", len(forecast_df))
print("Start date:", forecast_df["order_date"].min())
print("End date:", forecast_df["order_date"].max())
print("Unique dates:", forecast_df["date"].nunique())


Forecasting rows: 700232
Start date: 2024-02-03 04:31:58
End date: 2026-02-02 16:03:02
Unique dates: 731


## 6. Overall Demand EDA

We first examine total completed-order demand at the daily level and check whether demand differs by day of week.


In [5]:
daily_demand = (
    forecast_df
    .groupby("date")["quantity"]
    .sum()
    .reset_index()
)

print("Average daily demand:", round(daily_demand["quantity"].mean(), 2))
print("\nDemand statistics:")
print(daily_demand["quantity"].describe())

fig = px.line(
    daily_demand,
    x="date",
    y="quantity",
    title="Daily E-Commerce Demand",
    labels={"date": "Date", "quantity": "Units Sold"},
    template="plotly_white"
)

fig.update_traces(
    hovertemplate="Date: %{x}<br>Demand: %{y:,.0f} units<extra></extra>"
)

fig.update_layout(title_x=0.5)
fig.show()


Average daily demand: 2871.48

Demand statistics:
count     731.000000
mean     2871.476060
std       106.388589
min      1963.000000
25%      2803.500000
50%      2870.000000
75%      2940.000000
max      3240.000000
Name: quantity, dtype: float64


In [23]:
daily_demand["day_name"] = daily_demand["date"].dt.day_name()

weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

weekday_avg = (
    daily_demand
    .groupby("day_name")["quantity"]
    .mean()
    .reindex(weekday_order)
    .reset_index()
)

weekday_avg.columns = ["day_name", "average_demand"]

fig = px.bar(
    weekday_avg,
    x="day_name",
    y="average_demand",
    color="day_name",
    text="average_demand",
    title="Average Demand by Day of Week",
    template="plotly_white"
)


fig.update_layout(
    title_x=0.5,
    showlegend=False
)

fig.show()


## 7. Category & Product Analysis

We compare total completed-order demand across categories and identify high-demand products.


In [24]:
category_demand = (
    forecast_df
    .groupby("category")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

print("Total Demand by Category:")
print(category_demand)

fig = px.bar(
    category_demand,
    x="category",
    y="quantity",
    color="category",
    text="quantity",
    title="Total Demand by Category",
    template="plotly_white"
)


fig.update_layout(
    title_x=0.5,
    xaxis_title="Category",
    yaxis_title="Total Units Sold",
    showlegend=False
)

fig.show()


Total Demand by Category:
      category  quantity
0       Sports    421071
1  Electronics    420837
2         Home    420362
3     Clothing    418837
4       Health    417942


In [8]:
product_demand = (
    forecast_df
    .groupby(["product_name", "category"])["quantity"]
    .sum()
    .reset_index()
    .sort_values("quantity", ascending=False)
)

top_products = product_demand.head(10)

print("Top 10 Products by Demand:")
print(top_products)

fig = px.bar(
    top_products,
    x="quantity",
    y="product_name",
    color="category",
    text="quantity",
    orientation="h",
    title="Top 10 Products by Total Demand",
    template="plotly_white"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside"
)

fig.update_layout(
    title_x=0.5,
    xaxis_title="Total Units Sold",
    yaxis_title="Product",
    yaxis={"categoryorder": "total ascending"}
)

fig.show()


Top 10 Products by Demand:
              product_name  category  quantity
42  Water Bottle Insulated    Sports     47546
40    Vacuum Robot Cleaner      Home     47447
19              Gym Gloves    Sports     47166
17             Foam Roller    Health     47127
8      Cotton T-Shirt Pack  Clothing     47102
34         Storage Box Set      Home     47065
37       Treadmill Folding    Sports     47046
29    Resistance Bands Set    Health     47039
3             Baseball Cap  Clothing     46915
36       Tennis Balls Pack    Sports     46909


## 8. Product Selection

A high-demand product with sufficient historical observations is selected for focused forecasting.

**Selected product: Water Bottle Insulated**


In [9]:
selected_product = "Water Bottle Insulated"

product_df = forecast_df[
    forecast_df["product_name"] == selected_product
].copy()

product_daily = (
    product_df
    .groupby("date")["quantity"]
    .sum()
    .reset_index()
    .sort_values("date")
    .reset_index(drop=True)
)

print("Selected product:", selected_product)
print("Historical days:", product_daily["date"].nunique())
print("Start date:", product_daily["date"].min())
print("End date:", product_daily["date"].max())


Selected product: Water Bottle Insulated
Historical days: 731
Start date: 2024-02-03 00:00:00
End date: 2026-02-02 00:00:00


## 9. Product-Level Daily Demand

Daily demand for the selected product is visualized before aggregation.


In [10]:
fig = px.line(
    product_daily,
    x="date",
    y="quantity",
    title=f"Daily Demand — {selected_product}",
    labels={"date": "Date", "quantity": "Units Sold"},
    template="plotly_white"
)

fig.update_traces(
    hovertemplate="Date: %{x}<br>Demand: %{y:,.0f} units<extra></extra>"
)

fig.update_layout(title_x=0.5)
fig.show()


## 10. Daily → Weekly Demand

Daily demand is noisy, so we aggregate it into weekly demand. The first and last weekly bins are removed because the dataset boundaries do not represent complete weeks.


In [11]:
weekly_df = (
    product_daily
    .set_index("date")
    .resample("W")["quantity"]
    .sum()
    .reset_index()
)

weekly_df_clean = (
    weekly_df
    .iloc[1:-1]
    .copy()
    .reset_index(drop=True)
)

print("Original weekly observations:", len(weekly_df))
print("Complete weekly observations:", len(weekly_df_clean))

fig = px.line(
    weekly_df_clean,
    x="date",
    y="quantity",
    markers=True,
    title=f"Weekly Demand — {selected_product}",
    labels={"date": "Week", "quantity": "Weekly Demand"},
    template="plotly_white"
)

fig.update_traces(
    hovertemplate="Week: %{x}<br>Demand: %{y:,.0f} units<extra></extra>"
)

fig.update_layout(title_x=0.5)
fig.show()


Original weekly observations: 106
Complete weekly observations: 104


## 11. Feature Engineering

The Random Forest uses calendar, lag, and rolling-average features.

- `lag_1`: previous week's demand
- `lag_2`: demand two weeks earlier
- `lag_4`: demand four weeks earlier
- `lag_8`: demand eight weeks earlier
- rolling means: recent historical demand averages

`shift(1)` is applied before rolling calculations so the current week's target is not used to create its own features.


In [12]:
model_df = weekly_df_clean.copy()

# Calendar features
model_df["week"] = model_df["date"].dt.isocalendar().week.astype(int)
model_df["month"] = model_df["date"].dt.month
model_df["quarter"] = model_df["date"].dt.quarter

# Lag features
model_df["lag_1"] = model_df["quantity"].shift(1)
model_df["lag_2"] = model_df["quantity"].shift(2)
model_df["lag_4"] = model_df["quantity"].shift(4)
model_df["lag_8"] = model_df["quantity"].shift(8)

# Rolling averages using only previous observations
model_df["rolling_mean_4"] = (
    model_df["quantity"].shift(1).rolling(4).mean()
)

model_df["rolling_mean_8"] = (
    model_df["quantity"].shift(1).rolling(8).mean()
)

model_df = model_df.dropna().reset_index(drop=True)

features = [
    "week",
    "month",
    "quarter",
    "lag_1",
    "lag_2",
    "lag_4",
    "lag_8",
    "rolling_mean_4",
    "rolling_mean_8"
]

print("Modeling observations:", len(model_df))
print("Features:", features)


Modeling observations: 96
Features: ['week', 'month', 'quarter', 'lag_1', 'lag_2', 'lag_4', 'lag_8', 'rolling_mean_4', 'rolling_mean_8']


## 12. Time-Based Train/Test Split

Time series must be split chronologically. We train on the earlier 80% and test on the later 20%.

We do **not** shuffle the observations because future information must not enter the training period.


In [13]:
X = model_df[features]
y = model_df["quantity"]

split_index = int(len(model_df) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print(
    "Training period:",
    model_df["date"].iloc[:split_index].min(),
    "to",
    model_df["date"].iloc[:split_index].max()
)

print(
    "Testing period:",
    model_df["date"].iloc[split_index:].min(),
    "to",
    model_df["date"].iloc[split_index:].max()
)


Training rows: 76
Testing rows: 20
Training period: 2024-04-07 00:00:00 to 2025-09-14 00:00:00
Testing period: 2025-09-21 00:00:00 to 2026-02-01 00:00:00


## 13. Baseline vs Random Forest

The baseline predicts the next week's demand using the previous week's demand (`lag_1`).

Random Forest then learns relationships among the calendar, lag, and rolling features.


In [14]:
# Baseline
baseline_predictions = X_test["lag_1"]

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_predictions)
)

# Random Forest
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

rf_mape = np.mean(
    np.abs((y_test - rf_predictions) / y_test)
) * 100

print("Baseline MAE :", round(baseline_mae, 2))
print("Baseline RMSE:", round(baseline_rmse, 2))

print("\nRandom Forest")
print("MAE :", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("R²  :", round(rf_r2, 3))
print("MAPE:", round(rf_mape, 2), "%")


Baseline MAE : 59.1
Baseline RMSE: 67.92

Random Forest
MAE : 35.16
RMSE: 42.94
R²  : 0.066
MAPE: 7.94 %


In [15]:
comparison_df = pd.DataFrame({
    "Actual": y_test.values,
    "Baseline": baseline_predictions.values,
    "Random Forest": rf_predictions
}, index=model_df["date"].iloc[split_index:])

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=comparison_df.index,
        y=comparison_df["Actual"],
        mode="lines+markers",
        name="Actual"
    )
)

fig.add_trace(
    go.Scatter(
        x=comparison_df.index,
        y=comparison_df["Random Forest"],
        mode="lines+markers",
        name="Random Forest"
    )
)

fig.update_layout(
    title="Actual vs Random Forest Weekly Demand",
    title_x=0.5,
    xaxis_title="Week",
    yaxis_title="Units",
    template="plotly_white"
)

fig.show()


## 14. Four-Week Demand Forecast

The final Random Forest is retrained on all available modeling observations.

Forecasting is recursive: after predicting one future week, that prediction becomes available as a lag for the following week.


In [16]:
final_rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

final_rf.fit(X, y)

history = model_df[["date", "quantity"]].copy()
future_predictions = []
last_date = history["date"].max()

for _ in range(4):

    future_date = last_date + pd.Timedelta(weeks=1)
    values = history["quantity"].tolist()

    future_features = pd.DataFrame([{
        "week": int(future_date.isocalendar().week),
        "month": future_date.month,
        "quarter": future_date.quarter,
        "lag_1": values[-1],
        "lag_2": values[-2],
        "lag_4": values[-4],
        "lag_8": values[-8],
        "rolling_mean_4": np.mean(values[-4:]),
        "rolling_mean_8": np.mean(values[-8:])
    }])

    prediction = final_rf.predict(
        future_features[features]
    )[0]

    prediction = max(0, prediction)

    future_predictions.append({
        "date": future_date,
        "forecast_demand": prediction
    })

    history = pd.concat([
        history,
        pd.DataFrame({
            "date": [future_date],
            "quantity": [prediction]
        })
    ], ignore_index=True)

    last_date = future_date

forecast_4_weeks = pd.DataFrame(future_predictions)

forecast_4_weeks["forecast_demand"] = (
    forecast_4_weeks["forecast_demand"]
    .round()
    .astype(int)
)

total_4_week_forecast = int(
    forecast_4_weeks["forecast_demand"].sum()
)

print(forecast_4_weeks)
print("\n4-Week Forecast:", total_4_week_forecast, "units")


        date  forecast_demand
0 2026-02-08              439
1 2026-02-15              449
2 2026-02-22              468
3 2026-03-01              460

4-Week Forecast: 1816 units


## 15. Inventory Optimization

We calculate:
- average daily demand,
- demand variability,
- average delivery lead time,
- safety stock,
- reorder point,
- suggested replenishment.

A z-score of **1.65** is used for the safety-stock calculation.


In [17]:
average_daily_demand = product_daily["quantity"].mean()
daily_demand_std = product_daily["quantity"].std()
average_lead_time = product_df["delivery_days"].mean()

z_score = 1.65

safety_stock = round(
    z_score
    * daily_demand_std
    * np.sqrt(average_lead_time)
)

lead_time_demand = (
    average_daily_demand
    * average_lead_time
)

reorder_point = round(
    lead_time_demand + safety_stock
)

# Use the most recent product record for current stock.
product_info = product_df.sort_values("order_date")
latest_row = product_info.iloc[-1]

latest_stock = latest_row["stock_quantity"]
latest_inventory_date = latest_row["order_date"]

required_inventory = (
    total_4_week_forecast
    + safety_stock
)

replenishment_quantity = max(
    0,
    round(required_inventory - latest_stock)
)

print("Average daily demand:", round(average_daily_demand, 2))
print("Average lead time:", round(average_lead_time, 2), "days")
print("Safety stock:", safety_stock, "units")
print("Reorder point:", reorder_point, "units")
print("Latest inventory date:", latest_inventory_date)
print("Current stock:", latest_stock, "units")
print("Suggested replenishment:", replenishment_quantity, "units")


Average daily demand: 65.04
Average lead time: 7.51 days
Safety stock: 71 units
Reorder point: 559 units
Latest inventory date: 2026-02-02 15:38:53
Current stock: 133 units
Suggested replenishment: 1754 units


In [18]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=forecast_4_weeks["date"],
        y=forecast_4_weeks["forecast_demand"],
        text=forecast_4_weeks["forecast_demand"],
        textposition="outside",
        name="Forecast Demand",
        hovertemplate="Week: %{x}<br>Forecast: %{y:,.0f} units<extra></extra>"
    )
)

fig.add_hline(
    y=reorder_point,
    line_dash="dash",
    annotation_text=f"Reorder Point: {reorder_point}"
)

fig.add_hline(
    y=latest_stock,
    line_dash="dot",
    annotation_text=f"Current Stock: {latest_stock}"
)

fig.update_layout(
    title=f"Inventory Planning — {selected_product}",
    title_x=0.5,
    xaxis_title="Week",
    yaxis_title="Units",
    template="plotly_white"
)

fig.show()


## 16. Final Business Recommendation

The model forecast is converted into an operational inventory decision.

**Decision rule:** if current stock is below the reorder point, reorder.


In [19]:
if latest_stock <= reorder_point:
    recommendation = "REORDER NOW"
else:
    recommendation = "Stock is above reorder point"

print("=" * 55)
print("E-COMMERCE DEMAND FORECASTING")
print("=" * 55)

print("Product:", selected_product)

print("\nMODEL PERFORMANCE")
print("MAE :", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("MAPE:", round(rf_mape, 2), "%")

print("\nFORECAST")
print("Next 4 weeks:", total_4_week_forecast, "units")

print("\nINVENTORY")
print("Average daily demand:", round(average_daily_demand, 2))
print("Lead time:", round(average_lead_time, 2), "days")
print("Safety stock:", safety_stock, "units")
print("Reorder point:", reorder_point, "units")
print("Current stock:", latest_stock, "units")
print("Suggested replenishment:", replenishment_quantity, "units")

print("\nRECOMMENDATION")
print(recommendation)

print("=" * 55)


E-COMMERCE DEMAND FORECASTING
Product: Water Bottle Insulated

MODEL PERFORMANCE
MAE : 35.16
RMSE: 42.94
MAPE: 7.94 %

FORECAST
Next 4 weeks: 1816 units

INVENTORY
Average daily demand: 65.04
Lead time: 7.51 days
Safety stock: 71 units
Reorder point: 559 units
Current stock: 133 units
Suggested replenishment: 1754 units

RECOMMENDATION
REORDER NOW


# Conclusion

This project combines demand forecasting with an inventory-planning decision.

The final workflow is:

**Completed orders → demand analysis → product selection → weekly aggregation → lag/rolling features → time-based validation → Random Forest → four-week forecast → safety stock → reorder point → replenishment recommendation.**

The notebook intentionally excludes earlier experimental daily models and repeated feature experiments so the final project remains focused, explainable, and portfolio-ready.
